In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import scipy.ndimage
import csv, os, gc
!pip install transformers -q
from transformers import ViTForImageClassification

SEED = 42
torch.manual_seed(SEED)
device = torch.device('cpu')
print(f"Device: {device}")

DRIVE    = '/content/drive/MyDrive/ViT-Robustness'
METRICS  = DRIVE + '/results/metrics'
os.makedirs(METRICS, exist_ok=True)

# Load test images in raw form (no normalization)
transform_raw = transforms.ToTensor()
test_set_raw  = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_raw)

# Load in small batches to avoid RAM spike
print("Loading test images...")
all_images, all_labels = [], []
loader = torch.utils.data.DataLoader(test_set_raw, batch_size=200, shuffle=False)
for imgs, lbls in loader:
    all_images.append(imgs)
    all_labels.append(lbls)
# (10000, 3, 32, 32) float32
all_images     = torch.cat(all_images)
all_labels     = torch.cat(all_labels)
true_labels_np = all_labels.numpy()
np.save(f'{METRICS}/true_labels.npy', true_labels_np)
print(f"Loaded: {all_images.shape}  Labels: {true_labels_np.shape}")

# Normalization
def normalize_cnn(imgs):
    mean = torch.tensor([0.4914,0.4822,0.4465]).view(1,3,1,1)
    std  = torch.tensor([0.2023,0.1994,0.2010]).view(1,3,1,1)
    return (imgs - mean) / std

def normalize_vit(imgs):
    # upsample only what is needed
    # NOT storing 224x224 for all 10k
    resized = torch.nn.functional.interpolate(
        imgs, size=(224,224), mode='bilinear', align_corners=False)
    mean = torch.tensor([0.5,0.5,0.5]).view(1,3,1,1)
    std  = torch.tensor([0.5,0.5,0.5]).view(1,3,1,1)
    return (resized - mean) / std

# Corruption functions, the batch
def corrupt_noise(imgs, sigma):
    return torch.clamp(imgs + torch.randn_like(imgs) * sigma, 0, 1)

def corrupt_blur(imgs, sigma):
    np_imgs  = imgs.numpy()
    blurred  = np.stack([scipy.ndimage.gaussian_filter(
                    img, sigma=[0, sigma, sigma]) for img in np_imgs])
    return torch.tensor(blurred, dtype=torch.float32)

def corrupt_brightness(imgs, scale):
    return torch.clamp(imgs * scale, 0, 1)

def corrupt_rotation(imgs, angle):
    np_imgs = imgs.permute(0,2,3,1).numpy()
    rotated = np.stack([scipy.ndimage.rotate(
                    img, angle, reshape=False) for img in np_imgs])
    return torch.tensor(rotated, dtype=torch.float32).permute(0,3,1,2)

corruptions = {
    'gaussian_noise': (corrupt_noise,       [0.25, 0.50, 0.75]),
    'gaussian_blur':  (corrupt_blur,        [1.0,  2.0,  4.0]),
    'brightness':     (corrupt_brightness,  [0.5,  0.3,  0.1]),
    'rotation':       (corrupt_rotation,    [15,   30,   45]),
}

# Inference in small batches
def run_inference(model, images_norm, is_vit=False, batch_size=128):
    """Run model on pre-normalized CPU tensor, return numpy predictions."""
    model.eval()
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(images_norm), batch_size):
            batch = images_norm[i:i+batch_size].to(device)
            if is_vit:
                out = model(pixel_values=batch).logits
            else:
                out = model(batch)
            all_preds.append(out.argmax(1).cpu().numpy())
            del batch, out
    return np.concatenate(all_preds)

# Load CNN
print("\nLoading CNN...")
cnn_model = models.resnet18(weights=None)
cnn_model.fc = torch.nn.Linear(cnn_model.fc.in_features, 10)
cnn_model.load_state_dict(torch.load(
    f'{DRIVE}/checkpoints/resnet18_best.pth', map_location=device))
cnn_model = cnn_model.to(device).eval()
print("CNN loaded.")

# ── Load ViT ───────────────────────────────────────────────────────────────
print("Loading ViT...")
vit_model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224', num_labels=10, ignore_mismatched_sizes=True)
vit_model.load_state_dict(torch.load(
    f'{DRIVE}/checkpoints/vit_best.pth', map_location=device))
vit_model = vit_model.to(device).eval()
print("ViT loaded.")

# ── Main evaluation loop ───────────────────────────────────────────────────
results = []

for corr_name, (fn, levels) in corruptions.items():
    for sev, level in enumerate(levels, start=1):
        print(f"\nCorrupting: {corr_name} severity {sev} (level={level})")

        # 1. Apply corruption at 32x32 — small tensors, low RAM
        corrupted = fn(all_images.clone(), level)
        corrupted = torch.clamp(corrupted, 0, 1)

        # 2. CNN: normalize at 32x32 and infer
        cnn_input = normalize_cnn(corrupted)
        cnn_preds = run_inference(cnn_model, cnn_input, is_vit=False)
        cnn_acc   = (cnn_preds == true_labels_np).mean()
        del cnn_input
        gc.collect()

        # 3. ViT: upsample + normalize in small chunks inside run_inference
        #    We do NOT store the full 224x224 dataset — normalize_vit is
        #    called inside the batch loop below
        vit_preds = []
        vit_model.eval()
        with torch.no_grad():
            for i in range(0, len(corrupted), 64):
                chunk = corrupted[i:i+64]               # (64, 3, 32, 32)
                chunk_vit = normalize_vit(chunk).to(device)  # upsample here
                out = vit_model(pixel_values=chunk_vit).logits
                vit_preds.append(out.argmax(1).cpu().numpy())
                del chunk, chunk_vit, out
        vit_preds = np.concatenate(vit_preds)
        vit_acc   = (vit_preds == true_labels_np).mean()

        # 4. Free corrupted tensor before next iteration
        del corrupted
        gc.collect()
        torch.cuda.empty_cache()

        print(f"  CNN={cnn_acc:.4f} | ViT={vit_acc:.4f}")

        results.append({
            'corruption':   corr_name,
            'severity':     sev,
            'level_value':  level,
            'cnn_accuracy': round(float(cnn_acc), 4),
            'vit_accuracy': round(float(vit_acc), 4),
        })

        # Save predictions
        np.save(f'{METRICS}/cnn_preds_{corr_name}_s{sev}.npy', cnn_preds)
        np.save(f'{METRICS}/vit_preds_{corr_name}_s{sev}.npy', vit_preds)

# ── Save CSV ───────────────────────────────────────────────────────────────
csv_path = f'{METRICS}/accuracy_table.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print(f"\nAll done! Results saved to {csv_path}")

Mounted at /content/drive
Device: cpu


100%|██████████| 170M/170M [00:02<00:00, 64.2MB/s]


Loading test images...
Loaded: torch.Size([10000, 3, 32, 32])  Labels: (10000,)

Loading CNN...
CNN loaded.
Loading ViT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([10, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


ViT loaded.

Corrupting: gaussian_noise severity 1 (level=0.25)
  CNN=0.2630 | ViT=0.1973

Corrupting: gaussian_noise severity 2 (level=0.5)
